# Nightingale: the deep ranker B1-DL, a small transformer (EXP-021), on Kaggle

The owner chose one deep model, **a small transformer** (decision D-12), and chose to train it on
**Kaggle** (D-7, 2026-10-13), because a Kaggle *commit* runs in the background: you can close the
browser once it has started. It **replaces logistic regression in the pipeline only if it is at
least as safe** (D-13), judged afterwards on the laptop by the rule recorded in `docs/08` (EXP-021)
before this notebook ever ran.

It trains two arms at the five seeds of `docs/05` §6 (42, 43, 44, 45, 46), ten networks:

| Arm | Encoder | Like |
|---|---|---|
| **B1-DL′+aug** | with the "asked" channel, 691 columns (`ddd14019c66eff10`) | B1-LR′+aug, the configured ranker |
| **B1-DL+aug** | without it, 607 columns (`3a0d5a5e01d7f427`) | B1-LR+aug; the protocol's A0 uses this arm |

Each arm uses exactly the rows its comparators used at that seed, stops early on XGBoost's held-out
10%, and is never tuned (T-6). Seed 42 is the model used. Then it scores every network at 100%, 50%
and 25% of the history on the recorded masks. **The test split is never downloaded.**

## How to run it (about 5 minutes of your time)

1. On kaggle.com: **Create → New Notebook**, then **File → Import Notebook** and upload this file
   (or paste its GitHub link).
2. In the right-hand panel, **Session options**:
   - **Accelerator: GPU T4 x2** (two GPUs: the notebook trains two seeds at once). P100 is not
     recommended: recent PyTorch builds no longer support it.
   - **Internet: on** (it clones the repository and downloads DDXPlus).
   - Kaggle allows GPUs and internet only on a phone-verified account (*Settings → Phone
     verification*).
3. Top right: **Save Version → Save & Run All (Commit) → Save.** That starts the background run.
   **You can close the browser now.**
4. Come back in about **2–4 hours** (Kaggle emails you when the version finishes). Open the
   notebook's latest version, go to **Output**, and download **`nightingale_b1_dl.zip`**. Unzip it
   on the laptop into `models/nightingale_b1_dl/` and tell Claude.

**If the run stops early** (a Kaggle limit, an error): open the notebook, **Add Input → Your
Work → Notebook Output Files**, choose this notebook's last version, then **Save & Run All** again.
Finished arms are copied from that output and skipped. `STATUS.txt` in the output says what
finished.

Kaggle's free GPU time is about 30 hours a week; this run should use 2–4 of them. **Keep the
notebook private** (Kaggle's default): models trained on DDXPlus are never published (`docs/11`
§4). Nothing here needs a password or a token.


In [ ]:
# The commit Claude gave you. "master" works too: every run.json records the exact commit.
COMMIT = "master"
SEEDS = (42, 43, 44, 45, 46)
SCORE_REDUCED = True  # score every network at 50% and 25% too (CPU, about 30-60 min)

In [ ]:
import glob
import os
import shutil
import subprocess
import sys
import threading
import time
from pathlib import Path

REPO = Path("/tmp/nightingale")  # outside /kaggle/working, so no data ends up in the output
OUT = Path("/kaggle/working/nightingale_b1_dl")
LOGS = Path("/kaggle/working/logs")
OUT.mkdir(parents=True, exist_ok=True)
LOGS.mkdir(parents=True, exist_ok=True)


def run(*command, **kwargs):
    print("$", " ".join(map(str, command)), flush=True)
    subprocess.run([str(c) for c in command], check=True, **kwargs)


if not REPO.exists():
    run("git", "clone", "-q", "https://github.com/HarshRohila02/nightingale.git", REPO)
run("git", "checkout", "-q", COMMIT, cwd=REPO)
run("git", "log", "--oneline", "-1", cwd=REPO)

import torch

GPUS = torch.cuda.device_count()
print("torch", torch.__version__, "| GPUs:", [torch.cuda.get_device_name(i) for i in range(GPUS)])
assert GPUS > 0, "No GPU: Session options -> Accelerator -> GPU T4 x2, then Save & Run All again"

In [ ]:
# Resume: arms finished by an earlier version, added as input, are copied in and skipped.
copied = 0
for run_json in glob.glob("/kaggle/input/**/b1_dl_*/seed*/run.json", recursive=True):
    source = Path(run_json).parent
    target = OUT / source.parent.name / source.name
    if not (target / "run.json").exists():
        shutil.copytree(source, target, dirs_exist_ok=True)
        copied += 1
print(f"copied {copied} finished arm(s) from earlier outputs")

In [ ]:
# DDXPlus from Hugging Face, at the snapshot the laptop uses (docs/11 §4).
# test.csv is never fetched before Phase 4 (docs/05 §2).
from huggingface_hub import hf_hub_download

for name in ["release_evidences.json", "release_conditions.json", "train.csv", "validate.csv"]:
    hf_hub_download(
        "aai530-group6/ddxplus",
        name,
        repo_type="dataset",
        revision="2ad986acc1ec62fb4a94171acc43f4fdd5bfde53",
        local_dir=str(REPO / "data" / "raw" / "ddxplus"),
    )
# Decode the vocabulary, then keep the 13 chest-pain conditions of each split (task 1a).
if not (REPO / "data" / "interim" / "ddxplus_chestpain_train.parquet").exists():
    run(sys.executable, "scripts/decode_ddxplus.py", cwd=REPO)
    for split in ("train", "validate"):
        run(sys.executable, "scripts/build_ddxplus_chestpain.py", "--split", split, cwd=REPO)

In [ ]:
# Smoke test: 1% of the patients, one epoch, both arms, on one GPU. Checks the whole job in a few
# minutes and prints how long a full arm should take; nothing is saved. A failure stops the run
# here, before it spends GPU time.
run(sys.executable, "scripts/train_deep.py", "--device", "cuda", "--smoke", "--out", "/tmp/smoke",
    cwd=REPO, env={**os.environ, "CUDA_VISIBLE_DEVICES": "0"})

In [ ]:
# The real run: both arms at every seed, one seed at a time per GPU (seed 42 first on GPU 0).
# Each seed's log goes to /kaggle/working/logs. Finished arms are skipped.
status = {}


def worker(gpu, seeds):
    for seed in seeds:
        log = LOGS / f"seed{seed}.log"
        started = time.time()
        with open(log, "a", encoding="utf-8") as handle:
            result = subprocess.run(
                [sys.executable, "scripts/train_deep.py", "--device", "cuda",
                 "--seed", str(seed), "--out", str(OUT)],
                cwd=REPO, stdout=handle, stderr=subprocess.STDOUT,
                env={**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)},
            )
        minutes = (time.time() - started) / 60
        status[seed] = "done" if result.returncode == 0 else f"FAILED (exit {result.returncode})"
        print(f"GPU {gpu}: seed {seed} {status[seed]} after {minutes:.0f} min", flush=True)


workers = min(GPUS, 2)
queues = [list(SEEDS[i::workers]) for i in range(workers)]
threads = [threading.Thread(target=worker, args=(gpu, seeds)) for gpu, seeds in enumerate(queues)]
for thread in threads:
    thread.start()
for thread in threads:
    thread.join()
for seed in SEEDS:
    print(f"seed {seed}: {status.get(seed)}")
    if status.get(seed) != "done":
        print(open(LOGS / f"seed{seed}.log", encoding="utf-8").read()[-3000:])

In [ ]:
# 50% and 25% of the history, on the recorded masks (docs/05 §3.7): the script refuses to score
# if they do not reproduce their digests. CPU only. A failure here never loses the models.
if SCORE_REDUCED:
    for seed in SEEDS:
        arms = {"B1-DL+aug": OUT / "b1_dl_aug" / f"seed{seed}", "B1-DL′+aug": OUT / "b1_dl_asked_aug" / f"seed{seed}"}
        models = [arg for label, path in arms.items() if (path / "run.json").exists()
                  for arg in ("--model", f"{label}={path}:deep")]
        if not models:
            continue
        with open(LOGS / f"reduced_seed{seed}.log", "w", encoding="utf-8") as handle:
            result = subprocess.run(
                [sys.executable, "scripts/evaluate_reduced_evidence.py", "--no-graph", "--no-red-flags",
                 "--out", str(OUT / f"reduced_evidence_seed{seed}.json"), *models],
                cwd=REPO, stdout=handle, stderr=subprocess.STDOUT,
            )
        print(f"seed {seed}: reduced-evidence scoring exit code {result.returncode}", flush=True)

In [ ]:
# What finished, and the one file to download (Output tab: nightingale_b1_dl.zip).
lines = []
for arm in ("b1_dl_asked_aug", "b1_dl_aug"):
    for seed in SEEDS:
        done = (OUT / arm / f"seed{seed}" / "run.json").exists()
        lines.append(f"{arm} seed {seed}: {'done' if done else 'NOT DONE'}")
(OUT / "STATUS.txt").write_text("\n".join(lines) + "\n", encoding="utf-8")
print("\n".join(lines))
for log in LOGS.glob("*.log"):
    shutil.copy(log, OUT / log.name)
archive = shutil.make_archive("/kaggle/working/nightingale_b1_dl", "zip", OUT)
print(archive, f"{os.path.getsize(archive) / 1e6:.1f} MB")